In [ ]:
from pathlib import Path
from atlas.io import get_valid_slice_folders

In [ ]:
#testing data path
series_folder = Path(
    "/home/xcamra/projects/data4test/Section Set 6"
)
print(series_folder.exists())

In [ ]:
from atlas.io import extract_s_number

series_list, failed_folders = get_valid_slice_folders(series_folder)
# Assign initial storage indices in numeric ATLAS section order.
series_list = sorted(series_list, key=extract_s_number)

In [ ]:
print(f"\nFound {len(series_list)} series.")

for f in series_list:
    print(f)

In [ ]:
import json

import numpy as np
import tifffile as tiff
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree

from atlas.image_analysis import calculate_mask_roi, mask_low_and_saturation

from atlas.io import extract_s_number, get_metadata_for_stitched_tif
from atlas.stitching import (
    stitch_ATLAS_tiles,
    add_tile_overlap_columns,
    apply_transforms_and_stitch,
    build_adjacency_matrix_from_costs,
    build_transform_dict_from_mst,
    get_tiles_dataframe,
    match_tiles,
)

In [ ]:
from atlas.io import StitchedZarrStore

store = StitchedZarrStore.open(series_folder / "stitched_sections.zarr")

buffer_in_microns = 1

max_shift_in_pixles = 200

# iterate over all folders in the series
for raw_data_folder in series_list:
    # Iterate over all items in the folder
    for file in raw_data_folder.iterdir():  
        # Check if it's a file with the desired extension
        if file.is_file() and file.suffix == ".ve-mif":
            print(f"File with '.ve-mif' extension found: {file.name}")
            mif_file = file
            mif_metadict = get_metadata_for_stitched_tif(mif_file)
            # use naming convention to find the first tif
            first_tif_path = next(raw_data_folder.glob("*.tif"))
            print(first_tif_path)
            extracted_number = extract_s_number(first_tif_path)
            # Define the output file path
            output_tif_path = raw_data_folder.parent.joinpath(f"stitched_image_S_{extracted_number}.tiff")
            output_cc_path = raw_data_folder.parent.joinpath(f"phaseCC_stitching_S_{extracted_number}.csv")
            output_jason_path = raw_data_folder.parent.joinpath(f"transforms_S_{extracted_number}.json")

            if output_tif_path.exists():
                print(f"✅ Skipping: {output_tif_path.name} already exists.")
            else:
                print(f"🔄 Stitching image for S_{extracted_number}...")

                try:
                    stitched_img, mif_tile_df, transform_dict, stitching_metadata = stitch_ATLAS_tiles(
                        mif_file,
                        buffer_microns=buffer_in_microns,
                        max_shift_pixels=max_shift_in_pixles,
                    )

                    # Save the stitched image as a TIFF file
                    tiff.imwrite(output_tif_path, 
                                 np.flipud(stitched_img), 
                                 metadata={"axes": "YX", "AcquisitionMetadata": mif_metadict})
                    
                    metadata_path = output_tif_path.with_name(output_tif_path.stem + "_metadata.json")

                    with open(metadata_path, "w") as f:
                        json.dump(stitching_metadata, f, indent=4)

                    mif_tile_df.to_csv(output_cc_path, index=False)

                    # Convert NumPy arrays to lists for JSON compatibility
                    json_ready_dict = {k: v.tolist() for k, v in transform_dict.items()}

                    # Save to JSON file
                    with open(output_jason_path, "w") as f:
                        json.dump(json_ready_dict, f, indent=2)

                    # Store the same oriented pixels and metadata in Zarr.
                    store.write_section(
                        atlas_section_number=mif_metadict["atlas_section_number"],
                        image=np.flipud(stitched_img),
                        acquisition_metadata=mif_metadict,
                        stitching_metadata=stitching_metadata,
                        transforms=json_ready_dict,
                        source_folder=raw_data_folder,
                        tile_dataframe=mif_tile_df,
                    )
                        
                except Exception as e:
                    # Handle any error
                    print(f"Unexpected error with item {file}: {e}")
            

In [ ]:
import tifffile as tiff

In [ ]:
with tiff.TiffFile(series_folder.joinpath("stitched_image_S_2.tiff")) as tif:
    metadata = tif.shaped_metadata[0]["AcquisitionMetadata"]

for m in metadata.items():
    print(m)

## Read a stitched section from the Zarr store

Use the ATLAS section number (the `S` number) to retrieve its image and stored records. The section metadata gives the corresponding Z index in the shared `(z, y, x)` image array.

In [ ]:
from io import StringIO
import pandas as pd

# Select an ATLAS section number, for example S_2.
atlas_section_number = 2
section_key = f"S_{atlas_section_number}"
section_record = dict(store.sections[section_key].attrs)
z_index = section_record["z_index"]

# Read the image cropped to this section's original stitched shape.
section_image = store.read_section(atlas_section_number)
print(f"{section_key} is stored at z={z_index}")
print(f"Image shape: {section_image.shape}, dtype: {section_image.dtype}")

# Acquisition metadata comes from the TIFF; these records replace its
# metadata JSON and transforms JSON companion files.
acquisition_metadata = section_record["acquisition_metadata"]
stitching_metadata = section_record["stitching_metadata"]
transforms = section_record["transforms"]

# The former phaseCC_stitching CSV is stored as text and read as a DataFrame.
tile_table_csv = section_record["tile_table_csv"]
tile_df = pd.read_csv(StringIO(tile_table_csv)) if tile_table_csv else None

print("Acquisition metadata fields:", list(acquisition_metadata))
print("Stitching metadata fields:", list(stitching_metadata or {}))
print("Transform entries:", len(transforms or {}))
print("Tile table shape:", tile_df.shape if tile_df is not None else None)